In [3]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
import torch
from qwen_vl_utils import process_vision_info

W0716 15:45:18.987000 31068 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,  # Set False for 8-bit
    bnb_4bit_compute_dtype=torch.float16
)

In [5]:
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct", torch_dtype="auto", device_map="auto", quantization_config=bnb_config,
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [6]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
You have video processor config saved in `preprocessor.json` file which is deprecated. Video processor configs should be saved in their own `video_preprocessor.json` file. You can rename the file or load and save the processor back which renames it automatically. Loading from `preprocessor.json` will be removed in v5.0.


In [ ]:
import os, csv, re
image_folder = './data/train/cloth'
start_from = "03693_00.jpg"
csv_file = 'new_data.csv'
anomaly_file = 'anomaly.csv'
start_processing = False


# Loop through images
for filename in os.listdir(image_folder):
    if filename.lower().endswith(('.jpg')):
        if not start_processing:
            if filename == start_from:
                start_processing = True
                continue
            else:
                continue 
        
        path = os.path.join(image_folder, filename)
        img_name = os.path.basename(path)

        cloth_path = f"./data/train/cloth/{img_name}"
        image_path = f"./data/train/image/{img_name}"

        tagging_prompt = """Please tag the cloth in the image in terms of brand, sleeve, neckline, primary color, secondary color, and casuality.
        Examples: 1.Calvin Klein,long sleeve,v-neck,black,brown,formal 2.Levi's,short sleeve,round neck,blue,white,casual

        Note: If brand name is not present, please use "Unknown" as the brand name. Do not add new schema.
        Please use the following format: [tag1, tag2, tag3, ...].
        Do not include any other text in your response."""

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": cloth_path
                    },
                    {"type": "text", "text": tagging_prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")

        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        description_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        tag_text = description_text[0]

        del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()
        ###############################

        attr_prompt = """Please tag the person's attributes in the image in terms of fit, pants color, and hair color.
        Examples: 1.loose fit,red,black 2.tight fit,black,blonde

        Please use the following format: [tag1, tag2, tag3, ...].
        Do not include any other text in your response."""

        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image_path
                    },
                    {"type": "text", "text": attr_prompt},
                ],
            }
        ]

        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt",
        )
        inputs = inputs.to("cuda")


        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=128)

        generated_ids_trimmed = [
            out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        description_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )
        attribute_text = description_text[0]

        del inputs, image_inputs, video_inputs, generated_ids, generated_ids_trimmed
        torch.cuda.empty_cache()

        to_strip = "[]"
        tag_text = tag_text.strip(to_strip)
        attribute_text = attribute_text.strip(to_strip)
        data_text = img_name + "," + tag_text + "," + attribute_text
        cleaned_text = re.sub(r'[\[\]]', '', data_text)
        cleaned_text = re.sub(r',\s+', ',', cleaned_text)
        cleaned_text = cleaned_text.lower()

        row_list = cleaned_text.split(',')
        if len(row_list) == 10:
            with open(csv_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row_list)
        else:
            with open(anomaly_file, mode='a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row_list)


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Amlan\\Codes\\AIMS\\Fashionate-Old\\data\\train\\image\\03693_00.jpg'